1. 문서의 내용을 읽는다.
2. 문서를 쪼갠다.
    - 토큰 수 초과로 답변을 생성하지 못할 수 있고
    - 문서가 길면 (인풋이 길면) 답변 생성이 오래 걸린다.
3. 임베딩(벡터로 변환) -> 벡터 데이터베이스에 저장
4. 질문이 있을 때, 벡터 데이터베이스에서 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달

In [ ]:
# .docx (Word 문서) 파일에서 텍스트를 추출하는 라이브러리 설치
%pip install -qU docx2txt langchain_community

# 텍스트 분할을 위한 라이브러리 설치
%pip install -qU langchain-text-splitters

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
  # chunk_size = 청크 크기 (최대 문자 수)
  chunk_size=1000,
  # chunk_overlap = 인접한 청크끼리 겹치는 정도 (이전 내용 일부를 다음 청크에 포함)
  chunk_overlap=200
)

# .docx 파일에서 텍스트를 추출하여 문서 객체로 로드
loader = Docx2txtLoader("./tax_docs/tax_with_markdown.docx")
# 텍스트를 추출한 후, 텍스트 분할기를 사용하여 문서를 청크로 나누어 리스트 형태로 반환
document_list = loader.load_and_split(text_splitter=text_splitter)

In [ ]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

# OepnAI를 임베딩 하려면 환경변수에 OPENAI_API_KEY가 설정되어 있어야 한다
load_dotenv()

embedding = OpenAIEmbeddings(model="text-embedding-3-large")

In [ ]:
# Pinecone 설치
%pip install --upgrade --quiet \
    langchain-pinecone \
    langchain-openai  \
    langchain \
    pinecone-notebooks

In [ ]:
import os
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

index_name = 'tax-index'
pinecone_api_key = os.environ.get("PINECONE_API_KEY")

pc = Pinecone(api_key=pinecone_api_key)

database = PineconeVectorStore.from_documents(document_list, embedding, index_name=index_name)

In [78]:
query = '연봉 5천만원인 직장인의 종합소득세는?'

In [33]:
# 테이블의 경우 마크다운으로 변경을 해줘야 retriever가 잘 동작하고 답변도 잘 생성된다.
retriever = database.as_retriever(search_kwargs={"k": 3})
retriever.invoke(query)

[Document(id='5a95487b-9ab3-4507-8173-386bab29e036', metadata={'source': './tax_docs/tax_with_markdown.docx'}, page_content='[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n| 종합소득 과세표준          | 세율                                         |\n\n|-------------------|--------------------------------------------|\n\n| 1,400만원 이하     | 과세표준의 6퍼센트                             |\n\n| 1,400만원 초과     5,000만원 이하     | 84만원 + (1,400만원을 초과하는 금액의 15퍼센트)  |\n\n| 5,000만원 초과   8,800만원 이하     | 624만원 + (5,000만원을 초과하는 금액의 24퍼센트) |\n\n| 8,800만원 초과 1억5천만원 이하    | 3,706만원 + (8,800만원을 초과하는 금액의 35퍼센트)|\n\n| 1억5천만원 초과 3억원 이하         | 3,706만원 + (1억5천만원을 초과하는 금액의 38퍼센트)|\n\n| 3억원 초과    5억원 이하         | 9,406만원 + (3억원을 초과하는 금액의 38퍼센트)   |\n\n| 5억원 초과      10억원 이하    

In [ ]:
from langchain_openai import ChatOpenAI

# LLM 모델 초기화
llm = ChatOpenAI(model="gpt-5.4")

In [ ]:
# RetrieverQA Chain을 사용하여 효과적으로 검색하기
%pip install -U langchain langchainhub --quiet

In [ ]:
from dotenv import load_dotenv
from langsmith import Client

import os

load_dotenv()

LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
client = Client(api_key=LANGSMITH_API_KEY)
prompt = client.pull_prompt("rlm/rag-prompt", include_model=True)

In [35]:
from langchain_classic.chains import RetrievalQA

# RetrievalQA: 질문이 들어오면 (1)벡터DB에서 관련 문서 검색 → (2)프롬프트에 문서+질문 조합 → (3)LLM으로 답변 생성을 자동으로 처리하는 체인
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    # as_retriever()는 벡터DB를 통일된 Retriever 인터페이스로 변환 (Chroma, FAISS 등 DB 교체 시 코드 변경 최소화)
    retriever=database.as_retriever(),
    # LangSmith에서 가져온 RAG 프롬프트 템플릿을 체인에 적용
    chain_type_kwargs={"prompt": prompt}
)

In [36]:
# query -> 직장인 -> 거주자 chain 추가
ai_message = qa_chain({"query": query})

In [37]:
ai_message['result']

'연봉 4천만원을 과세표준 4천만원으로 보면, 소득세는 **84만원 + (4,000만원 - 1,400만원) × 15% = 474만원**입니다.  \n즉, **종합소득산출세액은 474만원**입니다.  \n다만 실제 연봉 기준 세금은 각종 공제·세액공제 등을 반영해야 해서 달라질 수 있습니다.'

In [79]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

dictionary = ["사람을 나타내는 표현 -> 거주자"]

prompt = ChatPromptTemplate.from_template(f"""
  사용자의 질문을 보고, 우리의 사전을 참고해서 사용자의 질문을 변경해주세요.
  만약 변경할 필요가 없다고 판단된다면, 사용자의 질문을 그대로 반환해주세요.
  사전: {dictionary}

  질문: {{question}}
"""
)

dictionary_chain = prompt | llm | StrOutputParser()

In [80]:
new_question = dictionary_chain.invoke({"question": query})

In [81]:
query

'연봉 5천만원인 직장인의 종합소득세는?'

In [82]:
new_question

'연봉 5천만원인 거주자의 종합소득세는?'

In [84]:
tax_chain = {"query": dictionary_chain} | qa_chain

In [86]:
ai_response = tax_chain.invoke({"question": query})

In [87]:
ai_response

{'query': '연봉 5천만원인 거주자의 종합소득세는?',
 'result': '과세표준이 5,000만원이라면 종합소득세 산출세액은 **624만원**입니다.  \n근거는 제55조에 따라 **1,400만원 초과 5,000만원 이하 = 84만원 + (5,000만원-1,400만원)의 15% = 84만원 + 540만원 = 624만원**입니다.  \n다만 이는 **공제·감면 등을 반영하지 않은 산출세액**입니다.'}